# Data Splitting DEV — 150 val, stratified by view + disease

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import h5py
import json
import random
from collections import defaultdict, Counter

# Paths
PROJECT_ROOT = Path.cwd().parent
TRAIN_DIR    = PROJECT_ROOT / "train"
IMAGES_DIR   = TRAIN_DIR / "images"
LABELS_DIR   = TRAIN_DIR / "labels"

JSON_DIR = Path.cwd()

# All image filenames
all_image_filenames = sorted([name for name in os.listdir(str(IMAGES_DIR)) if name.endswith('.h5')])
all_labeled_filenames = sorted([name.replace('_label', '') for name in os.listdir(str(LABELS_DIR)) if name.endswith('.h5')])
all_unlabeled_filenames = sorted([name for name in all_image_filenames if name not in set(all_labeled_filenames)])

print(f"Total images : {len(all_image_filenames)}")
print(f"Labeled      : {len(all_labeled_filenames)}")
print(f"Unlabeled    : {len(all_unlabeled_filenames)}")


In [ ]:
#  Extract view + disease info from ALL labeled images
VIEW_NAMES = {1: "4CH", 2: "LVOT", 3: "RVOT", 4: "3VT"}
CLS_NAMES  = ["VSD", "AV_sten", "AoHypo", "AoV_sten", "DORV", "PV_sten", "RAA"]

labeled_info = {}  # filename -> {view, label, has_disease, diseases}

for fn in all_labeled_filenames:
    img_path = os.path.join(str(IMAGES_DIR), fn)
    lbl_path = os.path.join(str(LABELS_DIR), fn.replace('.h5', '_label.h5'))

    with h5py.File(img_path, 'r') as f:
        view = int(np.array(f['view'][:]).reshape(-1)[0])

    with h5py.File(lbl_path, 'r') as f:
        label = f['label'][:]  # (7,) binary vector

    has_disease = int(label.sum()) > 0
    diseases = [CLS_NAMES[i] for i in range(7) if label[i] > 0]

    labeled_info[fn] = {
        'view': view,
        'label': label.tolist(),
        'has_disease': has_disease,
        'diseases': diseases,
    }

# Summary
print(f"Extracted info for {len(labeled_info)} labeled images\n")

# View distribution
view_counts = Counter(info['view'] for info in labeled_info.values())
for v in sorted(view_counts.keys()):
    n_dis = sum(1 for info in labeled_info.values() if info['view'] == v and info['has_disease'])
    n_nor = view_counts[v] - n_dis
    print(f"  View {v} ({VIEW_NAMES[v]}): {view_counts[v]:3d} total | {n_nor:3d} normal | {n_dis:3d} disease")

# Disease distribution
print(f"\nDisease distribution (total):")
for i, name in enumerate(CLS_NAMES):
    n = sum(1 for info in labeled_info.values() if info['label'][i] > 0)
    print(f"  {name:>10s}: {n}")
total_disease = sum(1 for info in labeled_info.values() if info['has_disease'])
total_normal  = len(labeled_info) - total_disease
print(f"\n  Normal: {total_normal} | Disease: {total_disease} | Total: {len(labeled_info)}")


### Stratified Split (view + disease balanced)

In [ ]:
#  Stratified split: ~150 validation, balanced by view AND disease
N_VAL_PER_VIEW = 37   # 37 × 4 = 148 (closest to 150 that's balanced)
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)

# Group filenames by (view, has_disease)
groups = defaultdict(list)
for fn, info in labeled_info.items():
    key = (info['view'], info['has_disease'])
    groups[key].append(fn)

print("Groups (view, has_disease)  count:")
for key in sorted(groups.keys()):
    v, dis = key
    label = "disease" if dis else "normal"
    print(f"  View {v} ({VIEW_NAMES[v]}) {label:>8s}: {len(groups[key])}")

# Strategy: for each view, take ~40% of disease cases + fill with normal to reach N_VAL_PER_VIEW
# This ensures good disease representation while keeping views balanced.

valid_filenames = []
for v in sorted(VIEW_NAMES.keys()):
    disease_pool = groups[(v, True)].copy()
    normal_pool  = groups[(v, False)].copy()
    random.shuffle(disease_pool)
    random.shuffle(normal_pool)

    # Take ~40% of disease cases for validation
    n_disease_val = max(1, min(len(disease_pool), round(len(disease_pool) * 0.40)))
    # Fill remaining with normal
    n_normal_val = N_VAL_PER_VIEW - n_disease_val
    n_normal_val = min(n_normal_val, len(normal_pool))

    val_disease = disease_pool[:n_disease_val]
    val_normal  = normal_pool[:n_normal_val]

    valid_filenames.extend(val_disease)
    valid_filenames.extend(val_normal)

    print(f"\n  View {v} ({VIEW_NAMES[v]}) validation: "
          f"{len(val_normal)} normal + {len(val_disease)} disease = {len(val_normal)+len(val_disease)}"
          f"  (pool: {len(normal_pool)} normal, {len(disease_pool)} disease)")

train_labeled_filenames = [fn for fn in all_labeled_filenames if fn not in set(valid_filenames)]

print(f"\n{'='*60}")
print(f"  Train labeled  : {len(train_labeled_filenames)}")
print(f"  Validation     : {len(valid_filenames)}")
print(f"  Train unlabeled: {len(all_unlabeled_filenames)}")
print(f"{'='*60}")

# Sanity checks
valid_set = set(valid_filenames)
train_set = set(train_labeled_filenames)
assert len(valid_set & train_set) == 0, "OVERLAP between train and valid!"
assert len(valid_set) + len(train_set) == len(all_labeled_filenames), "MISSING files!"
assert len(valid_filenames) == len(valid_set), "DUPLICATES in valid!"
print("\n No overlap, no duplicates, no missing files")

val_disease_count = sum(1 for fn in valid_filenames if labeled_info[fn]['has_disease'])
val_normal_count  = len(valid_filenames) - val_disease_count
print(f" Validation: {val_normal_count} normal + {val_disease_count} disease")


### Create JSONs

In [ ]:
def build_dataset_entry(images_dir, labels_dir, filename, has_label=True):
    image_h5_file_path = os.path.abspath(os.path.join(str(images_dir), filename))
    with h5py.File(image_h5_file_path, 'r') as f:
        view_id = int(np.array(f['view'][:]).reshape(-1)[0])

    entry = {
        "image": image_h5_file_path,
        "label": os.path.abspath(os.path.join(str(labels_dir),
                 filename.replace('.h5', '_label.h5'))) if has_label else None,
        "view_id": view_id,
    }
    return entry

valid_dataset           = [build_dataset_entry(IMAGES_DIR, LABELS_DIR, fn) for fn in valid_filenames]
train_labeled_dataset   = [build_dataset_entry(IMAGES_DIR, LABELS_DIR, fn) for fn in train_labeled_filenames]
train_unlabeled_dataset = [build_dataset_entry(IMAGES_DIR, LABELS_DIR, fn, has_label=False) for fn in all_unlabeled_filenames]

print(f"Generated datasets:")
print(f"  train_labeled   : {len(train_labeled_dataset)} entries")
print(f"  train_unlabeled : {len(train_unlabeled_dataset)} entries")
print(f"  valid           : {len(valid_dataset)} entries")

# Save
JSON_DIR.mkdir(parents=True, exist_ok=True)
for name, data in [("train_labeled.json", train_labeled_dataset),
                    ("train_unlabeled.json", train_unlabeled_dataset),
                    ("valid.json", valid_dataset)]:
    with open(JSON_DIR / name, "w") as f:
        json.dump(data, f, indent=4)
    print(f"  Saved: {JSON_DIR / name}")


In [ ]:
#  Detailed distribution tables
def detailed_distribution(filenames, labeled_info, title):
    print(f"\n{'='*65}")
    print(f"  {title}  ({len(filenames)} samples)")
    print(f"{'='*65}")

    view_counts = Counter()
    view_disease = Counter()
    disease_counts = Counter()

    for fn in filenames:
        info = labeled_info[fn]
        v = info['view']
        view_counts[v] += 1
        if info['has_disease']:
            view_disease[v] += 1
            for d in info['diseases']:
                disease_counts[d] += 1

    print(f"\n  {'View':<12s} {'Total':>6s} {'Normal':>8s} {'Disease':>8s} {'%Disease':>10s}")
    print(f"  {'-'*48}")
    for v in sorted(view_counts.keys()):
        n_dis = view_disease[v]
        n_nor = view_counts[v] - n_dis
        pct = 100 * n_dis / view_counts[v] if view_counts[v] > 0 else 0
        print(f"  {VIEW_NAMES[v]:<12s} {view_counts[v]:>6d} {n_nor:>8d} {n_dis:>8d} {pct:>9.1f}%")

    total = len(filenames)
    total_dis = sum(view_disease.values())
    print(f"  {'TOTAL':<12s} {total:>6d} {total-total_dis:>8d} {total_dis:>8d} {100*total_dis/total:>9.1f}%")

    if disease_counts:
        print(f"\n  Disease breakdown:")
        for d in CLS_NAMES:
            if disease_counts[d] > 0:
                print(f"    {d:>10s}: {disease_counts[d]}")

detailed_distribution(valid_filenames, labeled_info, "VALIDATION")
detailed_distribution(train_labeled_filenames, labeled_info, "TRAIN LABELED")

# Unlabeled view distribution
print(f"\n{'='*65}")
print(f"  TRAIN UNLABELED  ({len(train_unlabeled_dataset)} samples)")
print(f"{'='*65}")
ul_views = Counter(e['view_id'] for e in train_unlabeled_dataset)
for v in sorted(ul_views.keys()):
    print(f"  {VIEW_NAMES[v]:<12s} {ul_views[v]:>6d}")


### Preprocessing Stats

Preprocessing pipeline:
1. **Grayscale** → convierte a 1 canal
2. **Percentile clipping** (1%, 99%) → elimina outliers de intensidad  
3. **CLAHE** → adaptive equalization
4. **Z-score per-view** → normaliza a ~N(0,1) usando mean/std calculados SOLO de train_labeled

Esto le da al modelo inputs consistentes por vista, mejorando convergencia.

In [ ]:
import cv2
from skimage.color import rgb2gray

CLIP_P_LOW, CLIP_P_HIGH = 1, 99
CLAHE_CLIP_LIMIT, CLAHE_TILE = 2.0, 8

def _preprocess_no_zscore(raw_image):
    gray = rgb2gray(raw_image).astype(np.float32)
    lo, hi = np.percentile(gray, CLIP_P_LOW), np.percentile(gray, CLIP_P_HIGH)
    if hi - lo < 1e-6:
        return np.zeros_like(gray, dtype=np.float32)
    clip = np.clip((gray - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT,
                             tileGridSize=(CLAHE_TILE, CLAHE_TILE))
    return clahe.apply((clip * 255).astype(np.uint8)).astype(np.float32) / 255.0

view_pixels = defaultdict(list)

print(f"Computing stats from {len(train_labeled_dataset)} train_labeled images...")
for i, entry in enumerate(train_labeled_dataset):
    with h5py.File(entry["image"], "r") as f:
        raw = f["image"][:]
        view = int(np.array(f["view"][:]).reshape(-1)[0])
    view_pixels[view].append(_preprocess_no_zscore(raw).ravel())
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(train_labeled_dataset)}")

stats = {}
for v in sorted(view_pixels.keys()):
    all_px = np.concatenate(view_pixels[v])
    stats[v] = {
        "mean": float(np.mean(all_px)),
        "std": float(np.std(all_px)),
        "n_images": len(view_pixels[v]),
    }
    print(f"  View {v} ({VIEW_NAMES.get(v, '?')}): "
          f"mean={stats[v]['mean']:.6f}  std={stats[v]['std']:.6f}  "
          f"n={stats[v]['n_images']}")

stats_path = JSON_DIR / "preprocessing_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"\nSaved  {stats_path}")

del view_pixels


### Challenge Validation (500 images, no labels)

In [ ]:
VALID_CHALLENGE_DIR = PROJECT_ROOT / "valid" / "images"

challenge_filenames = sorted([
    name for name in os.listdir(str(VALID_CHALLENGE_DIR)) if name.endswith('.h5')
])
print(f"Challenge validation images found: {len(challenge_filenames)}")

# Build entries with tests
challenge_dataset = []
challenge_errors  = []

for fn in challenge_filenames:
    image_path = os.path.abspath(os.path.join(str(VALID_CHALLENGE_DIR), fn))
    try:
        with h5py.File(image_path, 'r') as f:
            assert 'image' in f, f"Missing 'image' key in {fn}"
            assert 'view' in f,  f"Missing 'view' key in {fn}"

            view_id = int(np.array(f['view'][:]).reshape(-1)[0])
            img = f['image'][:]

            assert view_id in [1, 2, 3, 4], f"Invalid view_id={view_id} in {fn}"
            assert img.ndim >= 2, f"Image has wrong ndim={img.ndim} in {fn}"
            assert img.shape[0] > 10 and img.shape[1] > 10, f"Image too small {img.shape} in {fn}"
            assert img.max() > img.min(), f"Constant image (corrupted?) in {fn}"

        challenge_dataset.append({
            "image": image_path,
            "label": None,
            "view_id": view_id,
        })
    except Exception as e:
        challenge_errors.append((fn, str(e)))

# Test results
print(f"\n{'='*50}")
print(f"  CHALLENGE VALIDATION TESTS")
print(f"{'='*50}")
print(f"   Passed: {len(challenge_dataset)} / {len(challenge_filenames)}")
if challenge_errors:
    print(f"   Failed: {len(challenge_errors)}")
    for fn, err in challenge_errors[:10]:
        print(f"     {fn}: {err}")
else:
    print(f"   All images passed validation checks")

# No overlap test
challenge_basenames = set(os.path.basename(e['image']) for e in challenge_dataset)
train_basenames     = set(os.path.basename(e['image']) for e in train_labeled_dataset)
unlabeled_basenames = set(os.path.basename(e['image']) for e in train_unlabeled_dataset)
valid_basenames     = set(os.path.basename(e['image']) for e in valid_dataset)

overlap_train = challenge_basenames & train_basenames
overlap_unlab = challenge_basenames & unlabeled_basenames
overlap_valid = challenge_basenames & valid_basenames

print(f"\n  Overlap with train_labeled:   {' ' + str(len(overlap_train)) if overlap_train else ' 0'}")
print(f"  Overlap with train_unlabeled: {' ' + str(len(overlap_unlab)) if overlap_unlab else ' 0'}")
print(f"  Overlap with our validation:  {' ' + str(len(overlap_valid)) if overlap_valid else ' 0'}")

# View distribution
challenge_views = Counter(e['view_id'] for e in challenge_dataset)
print(f"\n  View distribution:")
for v in sorted(challenge_views.keys()):
    print(f"    {VIEW_NAMES[v]:<6s}: {challenge_views[v]}")

# Combine with unlabeled
train_unlabeled_combined = train_unlabeled_dataset + challenge_dataset
print(f"\n  Unlabeled original : {len(train_unlabeled_dataset)}")
print(f"  Challenge added    : {len(challenge_dataset)}")
print(f"  Combined total     : {len(train_unlabeled_combined)}")

# Save
combined_path = JSON_DIR / "train_unlabeled.json"
with open(combined_path, "w") as f:
    json.dump(train_unlabeled_combined, f, indent=4)
print(f"\n  Saved: {combined_path}")

challenge_path = JSON_DIR / "valid_challenge.json"
with open(challenge_path, "w") as f:
    json.dump(challenge_dataset, f, indent=4)
print(f"  Saved: {challenge_path}")


### Visualization: Our Validation (with GT)

In [ ]:
import matplotlib.pyplot as plt

N_SHOW = 16  # 4 per view (2 normal + 2 disease)

show_indices = []
for v in sorted(VIEW_NAMES.keys()):
    v_indices = [i for i, fn in enumerate(valid_filenames) if labeled_info[fn]['view'] == v]
    v_disease = [i for i in v_indices if labeled_info[valid_filenames[i]]['has_disease']]
    v_normal  = [i for i in v_indices if not labeled_info[valid_filenames[i]]['has_disease']]
    random.seed(42)
    show_indices.extend(random.sample(v_disease, min(2, len(v_disease))))
    show_indices.extend(random.sample(v_normal, min(2, len(v_normal))))

show_indices = show_indices[:N_SHOW]
n = len(show_indices)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols * 2, figsize=(ncols * 6, nrows * 3.5))
if nrows == 1:
    axes = axes[np.newaxis, :]

for idx_pos, idx in enumerate(show_indices):
    fn = valid_filenames[idx]
    info = labeled_info[fn]
    entry = valid_dataset[idx]

    with h5py.File(entry['image'], 'r') as f:
        img = f['image'][:]
    with h5py.File(entry['label'], 'r') as f:
        mask = f['mask'][:]

    if img.ndim == 3:
        from skimage.color import rgb2gray as _rgb2gray
        img_gray = _rgb2gray(img)
    else:
        img_gray = img

    row = idx_pos // ncols
    col = (idx_pos % ncols) * 2

    # Image
    axes[row, col].imshow(img_gray, cmap='gray')
    view_name = VIEW_NAMES[info['view']]
    if info['has_disease']:
        disease_str = ", ".join(info['diseases'])
        title = f"{view_name} | {disease_str}"
        axes[row, col].set_title(title, fontsize=9, fontweight='bold', color='red')
    else:
        axes[row, col].set_title(f"{view_name} | Normal", fontsize=9, fontweight='bold', color='green')
    axes[row, col].axis('off')

    # GT Mask
    axes[row, col+1].imshow(mask, cmap='nipy_spectral', vmin=0, vmax=14)
    axes[row, col+1].set_title(f"GT ({len(np.unique(mask))} cls)", fontsize=9)
    axes[row, col+1].axis('off')

for i in range(n, nrows * ncols):
    row = i // ncols
    col = (i % ncols) * 2
    axes[row, col].axis('off')
    axes[row, col+1].axis('off')

plt.suptitle(f"Our Validation — {len(valid_filenames)} samples "
             f"({sum(1 for fn in valid_filenames if labeled_info[fn]['has_disease'])} disease)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


### Visualization: Challenge Validation (no GT)

In [ ]:
N_SHOW_CH = 16

show_ch_indices = []
for v in sorted(VIEW_NAMES.keys()):
    v_indices = [i for i, e in enumerate(challenge_dataset) if e['view_id'] == v]
    random.seed(123)
    show_ch_indices.extend(random.sample(v_indices, min(4, len(v_indices))))

show_ch_indices = show_ch_indices[:N_SHOW_CH]
n = len(show_ch_indices)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
if nrows == 1:
    axes = axes[np.newaxis, :]

for idx_pos, idx in enumerate(show_ch_indices):
    entry = challenge_dataset[idx]
    fn = os.path.basename(entry['image'])

    with h5py.File(entry['image'], 'r') as f:
        img = f['image'][:]

    if img.ndim == 3:
        from skimage.color import rgb2gray as _rgb2gray
        img_gray = _rgb2gray(img)
    else:
        img_gray = img

    row = idx_pos // ncols
    col = idx_pos % ncols

    axes[row, col].imshow(img_gray, cmap='gray')
    axes[row, col].set_title(f"{VIEW_NAMES[entry['view_id']]} | {fn[:20]}",
                              fontsize=8, fontweight='bold', color='blue')
    axes[row, col].axis('off')

for i in range(n, nrows * ncols):
    row = i // ncols
    col = i % ncols
    axes[row, col].axis('off')

plt.suptitle(f"Challenge Validation — {len(challenge_dataset)} images (NO labels)",
             fontsize=14, fontweight='bold', color='blue')
plt.tight_layout()
plt.show()

print(f"\n  Challenge images NO tienen GT ni class labels.")
print(f"    Solo se usan para: 1) unlabeled en semi-supervised, 2) inference/submission.")
